# Ejercicio 1 · Inspeccionar el payload antes de modificar su codificación

Selecciona el kernel **Python (lorawan11 .venv)**. Este notebook contiene una sola celda de código Python, independiente de los notebooks 07 y 08. Puedes ejecutarla después de leer la explicación.

El ejercicio trabaja con 160 ppb y un flip del bit 5. Las instrucciones de GNU poke al final se ejecutan en su terminal interactiva.


# Laboratorio de codificación del payload

Primer ejercicio: inspeccionar una medición de 160 ppb y entender un flip del
bit 5 antes de diseñar la representación propuesta por el asesor.

## Archivos del ejemplo

| Archivo | Contenido hexadecimal | Valor decodificado |
|---|---|---:|
| `samples/o3_160ppb.bin` | `01 02 00 A0` | 160 ppb |
| `samples/o3_160ppb_flip_bit5.bin` | `01 02 00 80` | 128 ppb |

Cada archivo contiene **cuatro bytes binarios**, no caracteres que escriban el
hexadecimal. Se generan con las funciones de `src/encoding/cayenne.py`.

Desde la raíz del repositorio puedes reproducirlos y verificar sus valores:

```bash
.venv/bin/python experiments/cayenne_mod/generar_ejemplo.py
```

Usamos el entorno del proyecto porque el paquete de codificación también
importa la dependencia criptográfica PyCryptodome.

El script comprueba los bytes esperados, la decodificación y que sólo cambie un
bit. Conserva las muestras existentes; si modificaste alguna, avisa en lugar
de sobrescribirla.

## Cómo leer el payload

| Desplazamiento en bytes, desde 0 | Hexadecimal | Significado |
|---:|---|---|
| 0 | `01` | Canal 1 |
| 1 | `02` | Tipo Analog Input |
| 2–3 | `00 A0` | Entero de 16 bits con signo, big-endian: 160 |

La tesis reinterpreta ese entero a **1 ppb por unidad**. Es una convención del
proyecto: Analog Input de CayenneLPP normalmente usa escala 0.01. Un
decodificador genérico no interpretaría automáticamente estos bytes como
160 ppb. Véase [la decisión de codificación](../../docs/02_codificacion.md).

El valor 160 se descompone en `128 + 32`: están encendidos los bits 7 y 5.
Numeramos los bits desde 0, empezando por el menos significativo del valor.
El bit 5 está en el último byte del campo, aunque el campo ocupa dos bytes.

```text
original: 00000000 10100000 = 160
máscara:  00000000 00100000 =  32
XOR:      00000000 10000000 = 128
```

XOR invierte la posición seleccionada. Aquí resta 32 porque el bit ya era 1.
Aplicar el mismo flip a 128 lo vuelve 160: el sentido depende del dato.

Usando el umbral experimental de 155 ppb, esta lectura pasa de estar por
encima a quedar por debajo. Eso demuestra un cruce local; para afirmar que se
oculta el cruce de toda la red hay que comprobar las demás lecturas.

Estas muestras son payloads **sin cifrar** para estudiar la representación.
No contienen la trama LoRaWAN completa y este ejercicio aún no ejecuta el
ataque sobre texto cifrado ni implementa la contramedida.

La guía de GNU poke continúa más abajo en este notebook.


## Comprobar el ejemplo en Python

La siguiente celda usa el codificador actual del repositorio, muestra los bytes originales y modificados y verifica que sólo cambie un bit. Trabaja en memoria y no sobrescribe las muestras. También comprueba si GNU poke está disponible en el entorno del kernel.


In [1]:
# Primer ejercicio: codificar 160 ppb e invertir el bit 5 del valor.
# Usa el kernel Python (lorawan11 .venv).
from pathlib import Path
import sys
import shutil

# Funciona desde la raíz del repo o desde la carpeta de este notebook.
actual = Path.cwd().resolve()
RAIZ = next(
    (p for p in (actual, *actual.parents)
     if (p / "src" / "encoding" / "cayenne.py").is_file()),
    None,
)
if RAIZ is None:
    raise RuntimeError("Abre el notebook dentro del repositorio de la tesis.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.encoding.cayenne import encode_o3, decode_o3, flip_bit

original = encode_o3(160)
modificado = flip_bit(original, 5)

assert original == bytes.fromhex("01 02 00 A0")
assert modificado == bytes.fromhex("01 02 00 80")
assert decode_o3(original) == 160
assert decode_o3(modificado) == 128
assert sum((a ^ b).bit_count() for a, b in zip(original, modificado)) == 1
assert flip_bit(modificado, 5) == original

for nombre, payload in [("Original", original), ("Flip del bit 5", modificado)]:
    bits_valor = " ".join(f"{byte:08b}" for byte in payload[2:])
    print(f"{nombre}: {payload.hex(' ').upper()} → {decode_o3(payload)} ppb")
    print(f"  Bits del valor: {bits_valor}")
print("160 = 128 + 32. El bit 5 estaba encendido; invertirlo resta 32.")
print("Segundo flip: se recuperan los mismos cuatro bytes del original.")
print("Verificación: cambia exactamente un bit.")

poke = shutil.which("poke")
print(f"GNU poke: {poke or 'no encontrado en el PATH de este kernel'}")
print(f"Raíz del repositorio para la terminal: {RAIZ}")


Original: 01 02 00 A0 → 160 ppb
  Bits del valor: 00000000 10100000
Flip del bit 5: 01 02 00 80 → 128 ppb
  Bits del valor: 00000000 10000000
160 = 128 + 32. El bit 5 estaba encendido; invertirlo resta 32.
Segundo flip: se recuperan los mismos cuatro bytes del original.
Verificación: cambia exactamente un bit.
GNU poke: no encontrado en el PATH de este kernel
Raíz del repositorio para la terminal: /home/kali-lab-jelb/MaestriaPCIC/Tesis/lorawan11-rama-detection


## Qué debes observar

| Caso | Payload hexadecimal | Valor en binario (16 bits) | ppb |
|---|---|---|---:|
| Original | `01 02 00 A0` | `00000000 10100000` | 160 |
| Bit 5 invertido | `01 02 00 80` | `00000000 10000000` | 128 |

El canal y el tipo se conservan. XOR invierte el bit 5; como estaba encendido, resta 32. Invertirlo de nuevo suma 32 y recupera el original.

Este caso usa la representación actual. El diseño de pesos repartidos será un ejercicio posterior.


# Ejercicio 1: inspeccionar los cuatro bytes

Las instrucciones siguientes se basan en el
[manual de GNU poke](https://www.jemarch.net/poke-4.0-manual/poke.html),
secciones «Poking Bytes», «From Bytes to Integers» y «Big and Little Endians».
Los comandos no se han ejecutado en este entorno: no se encontró el programa
`poke` en PATH al preparar el ejemplo.

## Abrir la muestra

En la terminal, desde la raíz del repositorio:

```bash
poke experiments/cayenne_mod/samples/o3_160ppb.bin
```

Los comandos siguientes se escriben **dentro de poke**, uno por uno:

```text
.set endian big
.set obase 16
dump :from 0#B :size 4#B
```

El volcado debe contener `01 02 00 a0` (puede agrupar los bytes como
`0102 00a0`). `#B` indica un desplazamiento o tamaño en bytes.

## Interpretar los campos

```text
.set obase 10
uint<8> @ 0#B
uint<8> @ 1#B
int<16> @ 2#B
```

Los valores numéricos esperados son **1**, **2** y **160**; poke puede añadir
anotaciones de tipo. `@` permite interpretar datos desde una posición del
archivo. El entero de 16 bits empieza en el desplazamiento 2.

Para ver sólo el valor como bits:

```text
.set obase 2
uint<16> @ 2#B
```

Busca el patrón `0000000010100000` (la presentación puede omitir ceros
iniciales). Cuenta las posiciones desde la derecha: ¿cuáles están encendidas?

## Comparar con el flip ya generado

Abre la segunda muestra desde la misma sesión:

```text
.file experiments/cayenne_mod/samples/o3_160ppb_flip_bit5.bin
.set endian big
.set obase 16
dump :from 0#B :size 4#B
.set obase 10
int<16> @ 2#B
```

Ahora debes ver `01 02 00 80` y el valor **128**. Sólo cambió el bit 5 del
valor; el canal, el tipo y el tamaño permanecen iguales.

Para salir:

```text
.exit
```

Antes de diseñar la modificación, explica con tus palabras: ¿por qué este
flip resta 32 en vez de sumar 32? ¿Qué ocurriría al repetirlo? Las respuestas
se pueden comprobar con el patrón binario, sin utilizar un modelo LSTM.


## Tus observaciones

Edita esta celda después de hacer el ejercicio:

- ¿Por qué `00 A0` representa 160?
Por la contitucion hexadecimal del binario. 00 representa a la primera parte de loos 16 bits, el A0 corresponde a la segunda parte, esto traducido de hexadecimal a binario es 10100000. 

- ¿En qué byte del payload está el bit 5 del valor?
en el A0. 

- ¿Por qué el flip resta en este caso?
Porque el bit ya estaba encendido, realizar un XOR o flipp a ese bit hace la operacion contraria, en ese caso apagar el flip, restar 32.

- ¿Qué diferencia hay entre bajar esta lectura de 155 ppb y ocultar el cruce de toda la red?

Basicamente seria que un dato de contingencia pasae a no ser contingencia, o incluso en sus otros casos. Causaria una contingencia en datos que no lo son.

**Mis notas:**

Esta bastante interesante el uso de poke, solo que los comandos aun no los entiendo totalmente, por lo que me gustaria mas que me explicaras que es lo que pasa con cada comando que se utiliza. Pero es bastante claro com es que cada bit si tiene bastante peso, pero lo interesante es que antes del A0 (de hexadecimal) tiene 00 osea podemos utilizaresos para bajar la significancia. 

**Salida de los comandos de gnu poke**

lorawan11-rama-detection on  main [!?] via 🐍 v3.14.7 
❯ poke experiments/cayenne_mod/samples/o3_160ppb.bin
     _____
 ---'   __\_______
            ______)  GNU poke 4.3
            __)
           __)
 ---._______)

Copyright (C) 2024 The poke authors.
License GPLv3+: GNU GPL version 3 or later.
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.

Powered by Jitter 0.7.312.
Perpetrated by Jose E. Marchesi.

For help, type ".help".
Type ".exit" to leave the program.
(poke) .set 
auto-map          endian            obase             omaps             pretty-print      
autoremap         error-on-warning  odepth            omode             prompt-maps       
doc-viewer        oacutoff          oindent           pager             tracer            
(poke) .set endian big
(poke) .set obase 16
(poke) dump :from 0#8 :size 4#B
76543210  0011 2233 4455 6677 8899 aabb ccdd eeff  0123456789ABCDEF
00000000: 0102 00a0                                ....
(poke) .set obase 10
(poke) uint<B> @ 0#B
<stdin>:1:6: error: syntax error: unexpected identifier
(poke) uint<8> @ 0#B
1UB
(poke) uint<8> @ 1#B
2UB
(poke) uint<16> @ 2#B
160UH
(poke) .set obase 2
(poke) uint<16> @ 2#B
0b0000000010100000UH
(poke) file experiments/cayenne_mod/samples/o3_160ppb_flip_bit5.bin
<stdin>:1:1: error: undefined variable 'file'                                
(poke) .file experiments/cayenne_mod/samples/o3_160ppb_flip_bit5.bin                         
(poke) .set endian big                                                                       
(poke) .set obase 16 
(poke) dump :from 0#B :size 4#B
76543210  0011 2233 4455 6677 8899 aabb ccdd eeff  0123456789ABCDEF
00000000: 0102 0080                                ....
(poke) .set obase 10
(poke) int<16> @ 2#B  
128H
(poke) 
